# 06 GroupNN Baseline

Minimal Group-Structured Neural Network baseline for the extended DL feature set.

Scope of this notebook:
- reuse `dataset_DL` through `src.dl.load_one`
- do not create group-specific `.npy` files
- run a smoke test on `normal / US / L=22`
- compare `L in {22, 60, 252}` for `normal / US`
- leave gating for a later iteration

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd

CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.dl import load_one
from src.eval.metrics import evaluate
from src.eval.phases import iter_phases
from src.models.dl import GroupNNModel
from src.models.dl.group_utils import build_feature_groups

RESULTS_DIR = PROJECT_ROOT / "results"
LOG_DIR = PROJECT_ROOT / "log" / "GroupNN"
CKPT_DIR = RESULTS_DIR / "best_dl_models"
RESULTS_DIR.mkdir(exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

LS = [22, 60, 252]
SEED = 42

## Feature Group Check

In [ ]:
splits = load_one("normal", "US", L=22, tier="extended")
groups = build_feature_groups(splits["feature_cols"])
print({name: len(idx) for name, idx in groups.items()})
for name, idx in groups.items():
    print(name, [splits["feature_cols"][i] for i in idx])

## Smoke Test: normal / US / L=22

In [ ]:
model = GroupNNModel(
    feature_cols=splits["feature_cols"],
    L=22,
    group_hidden=64,
    final_hidden=64,
    dropout=0.2,
    max_epochs=100,
    early_stop_patience=10,
    early_stop_metric="qlike",
    seed=SEED,
    verbose=True,
)

model.fit(
    splits["train_X"], splits["train_y"],
    splits["valid_X"], splits["valid_y"],
)

y_pred = model.predict(splits["test_X"])
metrics = evaluate(splits["test_y"], y_pred)

assert y_pred.shape == splits["test_y"].shape
assert np.isfinite(y_pred).all()
assert (y_pred > 0).all()

print(metrics)
display(model.history_df().tail())

## Checkpoint Round Trip

In [ ]:
ckpt_path = CKPT_DIR / "GroupNN_smoke_normal_US_extended_L22_static.pt"
model.save_checkpoint(ckpt_path, extra={"regime": "normal", "country": "US", "feature_set": "extended", "protocol": "static"})
loaded = GroupNNModel.from_checkpoint(ckpt_path)
loaded_pred = loaded.predict(splits["test_X"])
assert loaded_pred.shape == y_pred.shape
print(f"checkpoint ok: {ckpt_path}")

## Static L-grid: normal / US

In [ ]:
def fit_one_l(regime: str, country: str, L: int):
    splits_l = load_one(regime, country, L=L, tier="extended")
    model_l = GroupNNModel(
        feature_cols=splits_l["feature_cols"],
        L=L,
        group_hidden=64,
        final_hidden=64,
        dropout=0.2,
        max_epochs=100,
        early_stop_patience=10,
        early_stop_metric="qlike",
        seed=SEED,
        verbose=False,
    )
    model_l.fit(
        splits_l["train_X"], splits_l["train_y"],
        splits_l["valid_X"], splits_l["valid_y"],
    )
    row = {
        "model": "GroupNN",
        "regime": regime,
        "country": country,
        "feature_set": "extended",
        "L": L,
        "protocol": "static",
        "tuning_val_qlike": model_l.best_val_qlike_,
        "tuning_val_mse": model_l.best_val_mse_,
        "tuning_best_epoch": model_l.best_epoch_,
        "epochs_used": model_l.epochs_used_,
        "n_features": len(splits_l["feature_cols"]),
    }
    hist = model_l.history_df().assign(regime=regime, country=country, L=L, protocol="static")
    return row, hist

tuned_rows = []
history_rows = []
for L in LS:
    row, hist = fit_one_l("normal", "US", L)
    tuned_rows.append(row)
    history_rows.append(hist)
    print(row)

tuned_df = pd.DataFrame(tuned_rows)
best_idx = tuned_df["tuning_val_qlike"].idxmin()
tuned_df["is_best_L"] = False
tuned_df.loc[best_idx, "is_best_L"] = True
tuned_df.to_csv(RESULTS_DIR / "groupnn_tuned_L.csv", index=False, encoding="utf-8-sig")
pd.concat(history_rows, ignore_index=True).to_csv(LOG_DIR / "training_history_static.csv", index=False, encoding="utf-8-sig")
display(tuned_df)

## Final Static Test for Best L

In [ ]:
best = tuned_df.loc[tuned_df["is_best_L"]].iloc[0]
best_L = int(best["L"])
best_epochs = max(1, int(best["tuning_best_epoch"]))

splits_best = load_one("normal", "US", L=best_L, tier="extended")
X_train_final = np.concatenate([splits_best["train_X"], splits_best["valid_X"]], axis=0)
y_train_final = np.concatenate([splits_best["train_y"], splits_best["valid_y"]], axis=0)

final_model = GroupNNModel(
    feature_cols=splits_best["feature_cols"],
    L=best_L,
    group_hidden=64,
    final_hidden=64,
    dropout=0.2,
    max_epochs=best_epochs,
    early_stop_patience=best_epochs + 1,
    early_stop_metric="qlike",
    seed=SEED,
    verbose=False,
)
final_model.fit(X_train_final, y_train_final)
test_pred = final_model.predict(splits_best["test_X"])

test_meta = splits_best["test_meta"].copy()
test_meta.index = pd.to_datetime(test_meta["prediction_date"])

result_rows = []
for phase_name, mask, _color in iter_phases(test_meta, "normal"):
    if mask.sum() == 0:
        continue
    metrics = evaluate(splits_best["test_y"][mask], test_pred[mask])
    result_rows.append({
        "model": "GroupNN",
        "regime": "normal",
        "country": "US",
        "feature_set": "extended",
        "L": best_L,
        "protocol": "static",
        "phase": phase_name,
        **metrics,
        "tuning_val_qlike": float(best["tuning_val_qlike"]),
        "tuning_best_epoch": best_epochs,
        "final_epochs": final_model.epochs_used_,
        "n_features": len(splits_best["feature_cols"]),
        "is_best_L": True,
    })

result_df = pd.DataFrame(result_rows)
result_df.to_csv(RESULTS_DIR / "groupnn_results_extended.csv", index=False, encoding="utf-8-sig")

ckpt_path = CKPT_DIR / f"GroupNN_normal_US_extended_L{best_L}_static.pt"
final_model.save_checkpoint(ckpt_path, extra={"regime": "normal", "country": "US", "feature_set": "extended", "protocol": "static"})

display(result_df)
print(f"checkpoint: {ckpt_path}")